In [1]:
pc = 'FotR_ex'
import pandas as pd, os
import matplotlib.pyplot as plt

config = {
    'root_dir':{
        'laptop': '/home/migueljaraiz/anaconda3/repos/',
        'cluster': '/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_extended/outputs/',
        'pc_pro': '/home/ninjaraiz/anaconda3/repos/'
    },
    'folder_to_save':{
        'laptop': '/home/migueljaraiz/anaconda3/repos/GMM_TIFON/',
        'cluster': '/home/m.jaraiz/Documentos/GMM/GMM_TIFON/',
        'FotR_ex': './example_GMM/'
    },
    'pylom_path': {
        'laptop': '/home/migueljaraiz/anaconda3/repos/pyLowOrder',
        'cluster': '/home/m.jaraiz/repos/pyLowOrder',
        'pc_pro': '/home/ninjaraiz/anaconda3/repos/pyLowOrder'
    }   
}

for key in ['root_dir', 'pylom_path']:
    config[key]['FotR_ex'] = [path for _, path in config[key].items()]

features = ['dcp_ds_log', 'dcp2_ds_log'] # , 'gradrhox_log', 'gradTx_log'

In [2]:
def plot_case_0(case, features, n_clusters, stencil, sep, save:bool = False):
    scale = 7
    markersize_dcp = 2
    folder_name = '_'.join(features)
    df = pd.read_csv(os.path.join(config['folder_to_save'][pc], f'{folder_name}/sep_{sep}/c_{n_clusters}/s_{stencil}', 'df_data_complete.csv'), delimiter=';', index_col=0)
    def get_column_from_df(column, case, df):
        if isinstance(column, (list, tuple)):
            lista = []
            for col in column:
                lista.append(df.groupby(['aoa', 'mach']).get_group((df['aoa'].unique()[case], df['mach'].unique()[case]))[col])
            return lista
        
        if isinstance(column, str):
            serie = df.groupby(['aoa', 'mach']).get_group((df['aoa'].unique()[case], df['mach'].unique()[case]))[column]
            return serie
        
    [x, z, cp, clusters] = get_column_from_df(['x', 'z', 'cp', 'clusters_GMM'], case, df)
    list_features_log = get_column_from_df(features, case, df)
    
    fig, ax = plt.subplots(2, 1, figsize=(12, 2*6))
    # ax = ax.flatten()
    ax[0].scatter(
        x, z,
        c='black', s=1
    )
    ax00 = ax[0].twinx()
    ax00.scatter(
        x, list_features_log[0], c='blue', s=markersize_dcp
    )

    ax01 = ax[0].twinx()
    ax01.scatter(
        x, list_features_log[1], c='red', s=markersize_dcp
    )
    ax[0].set_ylim(bottom = z.min()*scale, top = z.max()*scale)
    # Poner un tercer eje a la izquierda con cp
    ax_cp = ax[0].twinx()
    ax_cp.scatter(
        x, cp, c='green', s=markersize_dcp
    )
    ax_cp.spines['left'].set_position(('outward', 60))
    ax_cp.spines['left'].set_color('green')
    ax_cp.tick_params(axis='y', colors='green')
    ax_cp.invert_yaxis()
    ax_cp.yaxis.set_label_position('left')
    ax_cp.yaxis.tick_left()
    ax_cp.spines['right'].set_visible(False)
    
    # Arreglar ejes de los twinx
    ax00.set_yscale('log')
    ax01.set_yscale('log')
    #separar ejes secundarios y poner del mismo color que los puntos
    ax00.spines['right'].set_position(('outward', 30))
    ax01.spines['right'].set_position(('outward', 90))
    ax00.spines['right'].set_color('blue')
    ax01.spines['right'].set_color('red')
    ax00.tick_params(axis='y', colors='blue')
    ax01.tick_params(axis='y', colors='red')
    ax[0].set_xlabel('x')
    ax[0].set_ylabel('z')
    ax00.set_ylabel(features[0], color='blue')
    ax01.set_ylabel(features[0], color='red')
    ax[0].set_title(f'Case {case}')

    ax[1].scatter(
        x, z, c='black', s=markersize_dcp, alpha=0.7)
    ax[1].set_xlabel('x')
    ax[1].set_ylabel('z')
    ax[1].set_ylim(z.min() - 0.1, z.max() + 0.1)
    ax10 = ax[1].twinx()
    ax10.scatter(
        x, cp, c=clusters, s=markersize_dcp, alpha=0.7, cmap='viridis')
    ax10.set_ylabel('cP')
    ax10.invert_yaxis()

    fig.suptitle(f'Case {case} - features: {features} - sep: {sep} - stencil: {stencil}')
    # bajar título para que no se solape con los ejes
    fig.subplots_adjust(top=0.9)
    if bool(save):
        fig.savefig(
            os.path.join(
                config['folder_to_save'][pc], f'{folder_name}/sep_{sep}/c_{n_clusters}/example_case{case}_s_{stencil}.png'
                )
            )
        plt.close('all')
    else:
        plt.show()
# def plot_case_1(case, features, n_clusters, stencil, sep):
#     scale = 7
#     markersize_dcp = 1
#     folder_name = '_'.join(features)
#     df = pd.read_csv(os.path.join(config['folder_to_save'][pc], f'{folder_name}/sep_{sep}/c_{n_clusters}/s_{stencil}', 'df_data_complete.csv'), delimiter=';', index_col=0)
#     def get_column_from_df(column, case, df):
#         if isinstance(column, (list, tuple)):
#             lista = []
#             for col in column:
#                 lista.append(df.groupby(['aoa', 'mach']).get_group((df['aoa'].unique()[case], df['mach'].unique()[case]))[col])
#             return lista
        
#         if isinstance(column, str):
#             serie = df.groupby(['aoa', 'mach']).get_group((df['aoa'].unique()[case], df['mach'].unique()[case]))[column]
#             return serie
        
#     [x, z, cp, clusters] = get_column_from_df(['x', 'z', 'cp', 'clusters_GMM'], case, df)
#     list_features_log = get_column_from_df(features, case, df)
    
    

In [3]:
for case in [36, 87, 72, 78]:
    plot_case_0(
        case = case, # 36, 87, 72, 78
        features=features,
        n_clusters = 2,
        stencil = 8,
        sep = 1,
        save = True
    )
    
    plot_case_0(
        case = case, # 36, 87, 72, 78
        features=features,
        n_clusters = 2,
        stencil = 11,
        sep = 1,
        save = True
    )